In [17]:
import pandas as pd
final = pd.read_parquet("../data/processed/modeling_table.parquet")
final_clean = final.dropna(subset=["is_delayed"]).copy()

In [18]:
majority_baseline = final_clean["is_delayed"].value_counts(normalize=True).max()
print(f"Majority-class baseline accuracy: {majority_baseline:.3f}")

Majority-class baseline accuracy: 0.721


In [19]:
route_rates = final_clean.groupby("route_short_name")["is_delayed"].transform("mean")
route_baseline_preds = (route_rates > 0.5).astype(int)
route_baseline_acc = (route_baseline_preds == final_clean["is_delayed"]).mean()
print(f"Route-based naive baseline accuracy: {route_baseline_acc:.3f}")

Route-based naive baseline accuracy: 0.724


In [20]:
final = pd.read_parquet("../data/processed/modeling_table.parquet")
final_clean = final.dropna(subset=["is_delayed"]).copy()

majority_baseline = final_clean["is_delayed"].value_counts(normalize=True).max()
print(f"Majority-class baseline accuracy: {majority_baseline:.3f}")

route_rates = final_clean.groupby("route_short_name")["is_delayed"].transform("mean")
route_baseline_preds = (route_rates > 0.5).astype(int)
route_baseline_acc = (route_baseline_preds == final_clean["is_delayed"]).mean()
print(f"Majority baseline: {majority_baseline:.4f}")
print(f"Route baseline: {route_baseline_acc:.4f}")
print(f"Fraction of rows flagged 'predicted delayed': {(route_rates > 0.5).mean():.4f}")
print(final_clean.groupby('route_short_name')['is_delayed'].mean().describe())

Majority-class baseline accuracy: 0.721
Majority baseline: 0.7207
Route baseline: 0.7237
Fraction of rows flagged 'predicted delayed': 0.0320
count    577.000000
mean       0.304363
std        0.144670
min        0.004827
25%        0.212190
50%        0.299718
75%        0.378194
max        1.000000
Name: is_delayed, dtype: float64


In [22]:
print((final_clean.groupby('route_short_name')['is_delayed'].mean() > 0.5).sum(), "routes now cross 50%")

49 routes now cross 50%


Baselines (23-date winter dataset): majority-class baseline is 72.1% accuracy (all-not-delayed) — winter-only delay rate is 27.9%, higher than the mixed 5-date sample's 22.3%, consistent with Phase 3's finding that winter trips run meaningfully more delayed than summer. A naive route-based baseline (predict delayed if a route's historical rate > 50%) reaches 72.4% — barely better than majority, and the underlying reason is the same as in the smaller sample: even with winter's higher delay rate, per-route rates still cluster well under 50% (mean 30.4%, 75th percentile 37.8%), with only 49 of 577 routes crossing the threshold. This confirms route needs to be used as a continuous/weighted signal in a real model, not a hard cutoff — the EDA finding (11-23x rate gap between metro and bus) is real, but no fixed threshold captures it well.

In [21]:
final = pd.read_parquet("../data/processed/modeling_table.parquet")
print(final["service_date"].nunique(), "dates,", len(final), "rows")
print(final["temperature_c"].describe())
print(final["is_delayed"].mean())

23 dates, 13283093 rows
count    1.328309e+07
mean    -1.967149e+00
std      5.081611e+00
min     -1.420000e+01
25%     -4.800000e+00
50%     -7.000000e-01
75%      1.500000e+00
max      7.500000e+00
Name: temperature_c, dtype: float64
0.279327824779203


Step 4.2: the date-based split

In [23]:
test_dates = ["2024-02-16", "2024-02-21", "2024-01-26", "2023-12-20", "2023-12-09"]  
# a mix: 2 weekday Feb dates, 1 weekday Jan, 1 weekday Dec, 1 Saturday — spread across the season, includes a weekend

train = final_clean[~final_clean["service_date"].isin(test_dates)]
test = final_clean[final_clean["service_date"].isin(test_dates)]

print(f"Train: {len(train)} rows across {train['service_date'].nunique()} dates")
print(f"Test: {len(test)} rows across {test['service_date'].nunique()} dates")
print(f"Test share: {len(test) / (len(train) + len(test)):.3f}")

Train: 10271444 rows across 18 dates
Test: 2989265 rows across 5 dates
Test share: 0.225


In [24]:
print("Train delay rate:", train["is_delayed"].mean())
print("Test delay rate:", test["is_delayed"].mean())
print("Train temp range:", train["temperature_c"].min(), "to", train["temperature_c"].max())
print("Test temp range:", test["temperature_c"].min(), "to", test["temperature_c"].max())

Train delay rate: 0.28526660905711015
Test delay rate: 0.2589215074608641
Train temp range: -14.2 to 7.5
Test temp range: -3.8 to 6.3


Test share (22.5%) lands right in the normal 20-25% range, good.

Delay rates are reasonably close (28.5% train vs 25.9% test), a small gap, not alarming, and explainable: test's temperature range (-3.8 to 6.3) is milder than train's full range (down to -14.2), so test skews slightly toward the "less severe" end of winter. That's worth noting rather than ignoring, since it means test set is evaluating the model on a somewhat easier subset of winter than the full training range covers.

Locking the split in: none of the 5 test dates dip below -3.8°C, so the model's performance on genuinely extreme cold (-10°C and below, which train does include) is untested. Worth swapping one test date for a colder one so the ranges overlap more fully

In [25]:
test_dates = ["2024-02-16", "2024-02-21", "2024-01-26", "2023-12-20", "2024-01-13"]

train = final_clean[~final_clean["service_date"].isin(test_dates)]
test = final_clean[final_clean["service_date"].isin(test_dates)]

print(f"Train: {len(train)} rows across {train['service_date'].nunique()} dates")
print(f"Test: {len(test)} rows across {test['service_date'].nunique()} dates")
print(f"Test share: {len(test) / (len(train) + len(test)):.3f}")
print("Train delay rate:", train["is_delayed"].mean())
print("Test delay rate:", test["is_delayed"].mean())
print("Train temp range:", train["temperature_c"].min(), "to", train["temperature_c"].max())
print("Test temp range:", test["temperature_c"].min(), "to", test["temperature_c"].max())

Train: 10279863 rows across 18 dates
Test: 2980846 rows across 5 dates
Test share: 0.225
Train delay rate: 0.28958216661058617
Test delay rate: 0.24396429738403125
Train temp range: -14.2 to 7.5
Test temp range: -3.8 to 6.3


In [26]:
my_routes = ["607", "627", "514"]
mine = final_clean[final_clean["route_short_name"].isin(my_routes)]

print(mine.groupby("route_short_name").agg(
    n_rows=("is_delayed", "size"),
    n_dates=("service_date", "nunique"),
    delay_rate=("is_delayed", "mean")
))
print("\nTotal rows across all 3 routes:", len(mine))

                  n_rows  n_dates  delay_rate
route_short_name                             
514                73580       23    0.503683
607               103595       23    0.329958
627                34982       20    0.418215

Total rows across all 3 routes: 212157


In [27]:
   date_temps = final_clean.groupby("service_date")["temperature_c"].mean()
   bins = pd.cut(date_temps, bins=range(-15, 10, 5))
   print(bins.value_counts().sort_index())

temperature_c
(-15, -10]    2
(-10, -5]     3
(-5, 0]       9
(0, 5]        8
Name: count, dtype: int64


In [28]:
   print(final_clean.groupby("service_date")["temperature_c"].agg(["min", "max", "mean"]).sort_values("min"))

               min  max       mean
service_date                      
2024-01-16   -14.2 -3.0 -12.648587
2023-12-06   -14.0 -6.3 -11.920842
2024-01-18   -12.3 -2.3  -7.095725
2024-01-08   -11.3 -2.2  -8.835223
2023-12-04    -9.8 -6.7  -8.019298
2024-02-06    -7.0 -2.3  -3.847973
2024-02-08    -6.1 -2.9  -4.114260
2023-12-01    -5.6 -0.7  -1.664809
2024-02-11    -4.8 -2.1  -3.865404
2024-02-13    -4.8  0.4  -0.425250
2024-01-26    -3.8  0.9  -0.645234
2023-12-12    -3.3 -2.2  -2.910105
2024-01-13    -2.0  0.6  -0.561544
2024-01-21    -1.5  2.3   0.950659
2024-01-10    -1.0  3.3   0.597298
2023-12-20    -0.8  2.3   1.270436
2024-02-01    -0.6  4.1   2.596152
2023-12-09    -0.5  1.0  -0.015998
2024-01-24     0.4  2.1   1.221167
2024-02-21     0.4  4.0   3.014132
2024-02-16     0.7  6.3   4.982218
2024-02-03     1.8  7.5   5.641345
2023-12-17     3.1  6.7   4.821211
